In [1]:


!pip install -q -U transformers scipy gradio

In [2]:
import numpy as np
import torch
import gradio as gr

from transformers import pipeline
from scipy.io.wavfile import write

In [3]:
print("Loading MusicGen model...")

music_generator = pipeline(
    "text-to-audio",
    model="facebook/musicgen-small"
)

print("MusicGen model loaded successfully!")

Loading MusicGen model...


config.json:   0%|          | 0.00/7.87k [00:00<?, ?B/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 2047), got 2048. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.36GB            

model.safetensors: downloading bytes:           |  0.00B            

...

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

MusicGen model loaded successfully!


In [4]:
prompt = "an upbeat lo-fi hip-hop track with a relaxed piano melody"

output = music_generator(
    prompt,
    forward_params={
        "do_sample": True
    }
)

sampling_rate = output["sampling_rate"]
audio_array = output["audio"]

if audio_array.ndim > 1:
    audio_array = audio_array[0]

print(
    "Generated:",
    len(audio_array) / sampling_rate,
    "seconds of audio."
)

Generated: 29.94 seconds of audio.


In [5]:
from IPython.display import Audio, display

display(
    Audio(
        audio_array,
        rate=sampling_rate
    )
)

In [6]:
audio_to_save = np.asarray(audio_array)

# Convert float audio into int16
audio_int16 = np.int16(
    np.clip(audio_to_save, -1, 1) * 32767
)

write(
    "generated_music.wav",
    sampling_rate,
    audio_int16
)

print("Music saved successfully as generated_music.wav")

Music saved successfully as generated_music.wav


In [ ]:
import gradio as gr
import numpy as np
from scipy.io.wavfile import write
import os
import time


# ============================================================
# MUSIC GENERATION FUNCTION
# ============================================================

def generate_music(prompt, duration, guidance_scale):

    if not prompt or not prompt.strip():
        raise gr.Error("Please enter a music description.")

    prompt = prompt.strip()

    print("\n" + "=" * 60)
    print("Generating Music...")
    print("Prompt:", prompt)
    print("Duration:", duration)
    print("Guidance:", guidance_scale)
    print("=" * 60)

    # MusicGen uses approximately 50 audio tokens per second
    max_tokens = int(duration * 50)

    start_time = time.time()

    output = music_generator(
        prompt,
        forward_params={
            "do_sample": True,
            "guidance_scale": guidance_scale,
            "max_new_tokens": max_tokens
        }
    )

    generation_time = time.time() - start_time

    sampling_rate = output["sampling_rate"]

    audio_array = np.asarray(output["audio"])

    # Handle mono/stereo output
    if audio_array.ndim > 1:
        audio_array = np.squeeze(audio_array)

    # Make sure audio is 1D
    audio_array = audio_array.reshape(-1)

    # Normalize audio
    max_value = np.max(np.abs(audio_array))

    if max_value > 0:
        audio_array = audio_array / max_value

    # Convert to int16
    audio_int16 = np.int16(
        np.clip(audio_array, -1, 1) * 32767
    )

    # Create output directory
    os.makedirs("generated_music", exist_ok=True)

    filename = (
        "generated_music/music_"
        + str(int(time.time()))
        + ".wav"
    )

    # Save audio
    write(
        filename,
        sampling_rate,
        audio_int16
    )

    actual_duration = len(audio_array) / sampling_rate

    status = f"""
### ✅ Music Generated Successfully

**Prompt:** {prompt}

**Duration:** {actual_duration:.2f} seconds

**Sampling Rate:** {sampling_rate} Hz

**Generation Time:** {generation_time:.2f} seconds

**File:** `{filename}`
"""

    return (
        (sampling_rate, audio_array),
        filename,
        status
    )


# ============================================================
# EXAMPLE PROMPTS
# ============================================================

examples = [
    ["an upbeat lo-fi hip-hop track with a relaxed piano melody"],
    ["a cheerful acoustic guitar folk tune"],
    ["an energetic electronic dance track with a heavy bassline"],
    ["a calm classical piano piece"],
    ["a cinematic emotional orchestral soundtrack"],
    ["a peaceful ambient music track with soft pads"],
    ["a futuristic cyberpunk electronic track"],
]


# ============================================================
# CUSTOM CSS
# ============================================================

custom_css = """

body {
    background: linear-gradient(
        135deg,
        #0f172a,
        #111827,
        #1e1b4b
    );
}

.gradio-container {
    max-width: 1200px !important;
    margin: auto !important;
}

.title-box {
    text-align: center;
    padding: 30px;
    border-radius: 20px;
    background: linear-gradient(
        135deg,
        #312e81,
        #4f46e5,
        #7c3aed
    );
    color: white;
    margin-bottom: 25px;
    box-shadow: 0 10px 35px rgba(0,0,0,0.35);
}

.title-box h1 {
    font-size: 42px;
    margin-bottom: 8px;
}

.title-box p {
    font-size: 18px;
    opacity: 0.9;
}

.card {
    border-radius: 18px !important;
    border: 1px solid rgba(255,255,255,0.12) !important;
    background: rgba(255,255,255,0.05) !important;
    padding: 20px !important;
}

.generate-btn {
    height: 60px !important;
    font-size: 20px !important;
    font-weight: bold !important;
}

.status-box {
    border-radius: 15px !important;
}

.footer {
    text-align: center;
    margin-top: 25px;
    opacity: 0.7;
}

"""


# ============================================================
# BUILD UI
# ============================================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="indigo",
        secondary_hue="violet"
    ),
    css=custom_css,
    title="MusicGen Studio"
) as demo:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="title-box">

        <h1>🎵 MusicGen Studio</h1>

        <p>
            Generate Novel Music using a Pretrained
            Open-Source Foundation Model
        </p>

        <p>
            <b>Powered by Meta AI MusicGen</b>
        </p>

    </div>
    """)


    # --------------------------------------------------------
    # MAIN LAYOUT
    # --------------------------------------------------------

    with gr.Row():

        # ====================================================
        # LEFT PANEL
        # ====================================================

        with gr.Column(
            scale=1,
            elem_classes="card"
        ):

            gr.Markdown(
                """
                ## 🎼 Create Your Music

                Describe the music you want in natural language.
                """
            )

            prompt = gr.Textbox(
                label="Music Description",
                placeholder=(
                    "Example: An emotional cinematic piano "
                    "piece with soft strings..."
                ),
                lines=5,
                value=(
                    "an upbeat lo-fi hip-hop track "
                    "with a relaxed piano melody"
                )
            )

            duration = gr.Slider(
                minimum=5,
                maximum=30,
                value=15,
                step=1,
                label="⏱️ Duration (seconds)"
            )

            guidance = gr.Slider(
                minimum=1,
                maximum=5,
                value=3,
                step=0.5,
                label="🎯 Prompt Guidance"
            )

            generate_btn = gr.Button(
                "🎵 GENERATE MUSIC",
                variant="primary",
                elem_classes="generate-btn"
            )


        # ====================================================
        # RIGHT PANEL
        # ====================================================

        with gr.Column(
            scale=1,
            elem_classes="card"
        ):

            gr.Markdown(
                """
                ## 🎧 Generated Music
                Your AI-generated composition will appear here.
                """
            )

            audio_output = gr.Audio(
                label="Generated Music",
                type="numpy"
            )

            download_file = gr.File(
                label="💾 Download WAV"
            )

            status_output = gr.Markdown(
                value="Waiting for your prompt...",
                elem_classes="status-box"
            )


    # --------------------------------------------------------
    # EXAMPLES
    # --------------------------------------------------------

    gr.Markdown(
        """
        ## ✨ Try These Prompts
        """
    )

    gr.Examples(
        examples=examples,
        inputs=prompt
    )


    # --------------------------------------------------------
    # HOW IT WORKS
    # --------------------------------------------------------

    gr.Markdown(
        """
        ## 🧠 How MusicGen Works

        **Text Prompt → Transformer → Audio Tokens → Music**

        MusicGen is already pretrained, so this experiment does
        **not require dataset collection, model training or
        fine-tuning**.

        The model generates music directly from the description
        provided by the user.
        """
    )


    # --------------------------------------------------------
    # GENERATE BUTTON EVENT
    # --------------------------------------------------------

    generate_btn.click(
        fn=generate_music,
        inputs=[
            prompt,
            duration,
            guidance
        ],
        outputs=[
            audio_output,
            download_file,
            status_output
        ]
    )


    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.HTML("""
    <div class="footer">

        🎵 MusicGen Studio |
        Foundation Model Experiment |
        Text → Music

    </div>
    """)


# ============================================================
# LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_663/921596592.py:193: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b3693be863acbec2f9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Generating Music...
Prompt: a cheerful acoustic guitar folk tune
Duration: 15
Guidance: 3


/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:797: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))



Generating Music...
Prompt: an upbeat lo-fi hip-hop track with a relaxed piano melody
Duration: 15
Guidance: 3


/usr/local/lib/python3.13/dist-packages/gradio/processing_utils.py:797: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
